In [45]:
import pandas as pd

expression_data = pd.read_csv('/Users/apple/Desktop/KLin_Group/Project_2024/data/Morpho_data/dataset/Scala/exon_data_top2000.csv', 
                               header=None, 
                               index_col=0)

gene_names = pd.read_csv('/Users/apple/Desktop/KLin_Group/Project_2024/data/Morpho_data/dataset/Scala/top2000GEX/2000cell_name.csv', 
                         header=None)

gene_list = gene_names[0].tolist()

expression_data.columns = gene_list

expression_data.index.name = 'Cell'

expression_data.to_csv('/Users/apple/Desktop/KLin_Group/Project_2024/data/Morpho_data/dataset/Scala/exon_data_top2000_name.csv')


Each data modality (transcriptomic, electrophysiology, and morphology in Patch-seq data) leads to a different notion of similarity between cells.

Integration and Analysis of Multimodal (Patch-seq data: transcriptomic, electrophysiology, and morphology) Data

In [9]:
bd = "/Users/apple/Desktop/KLin_Group/Project_2024/data/Morpho_data/dataset/Scala" # Base directory

In [11]:
import cajal.sample_swc
import cajal.swc
from os.path import join

In [ ]:
import cajal.sample_swc
import cajal.swc
from os.path import join

cajal.sample_swc.compute_icdm_all_geodesic(
    infolder=join(bd,'swc'),
    out_csv=join(bd,'geodesic_100_icdm.csv'),
    #out_node_types=join(bd,"geodesic_100_node_types.npy"),
    n_sample=100,
    num_processes=8,  # num_processes can be set to the number of cores on your machine
    preprocess=cajal.swc.preprocessor_geo([1,3,4])
)

100%|█████████▉| 644/645 [04:08<00:00,  2.60it/s]


[]

In [3]:
import cajal.run_gw

cajal.run_gw.compute_gw_distance_matrix(
    intracell_csv_loc=join(bd,'geodesic_100_icdm.csv'),
    gw_dist_csv_loc=join(bd,'geodesic_100_gw.csv'),
    num_processes=14  # num_processes can be set to the number of cores on your machine
)

  0%|          | 0/207690 [00:00<?, ?it/s]

(array([[ 0.        , 29.52619971, 41.28311529, ..., 20.98309885,
         43.42000149, 58.73233694],
        [29.52619971,  0.        , 31.64544716, ..., 30.68485975,
         56.00344758, 69.98424886],
        [41.28311529, 31.64544716,  0.        , ..., 38.99039977,
         63.61903904, 59.90897295],
        ...,
        [20.98309885, 30.68485975, 38.99039977, ...,  0.        ,
         40.59676189, 57.80768401],
        [43.42000149, 56.00344758, 63.61903904, ..., 40.59676189,
          0.        , 55.16991241],
        [58.73233694, 69.98424886, 59.90897295, ..., 57.80768401,
         55.16991241,  0.        ]]),
 None)

Next, we focus on the electrophysiology and gene expression data, along with the metadata for each cell.

We input morphological, transcriptomic(exon_data), electrophysiological(ephys_data) and metdata(experiment infomation).

In [12]:
import pandas as pd

exon_data = pd.read_csv(join(bd,'exon_data.csv'),index_col='cell id')
metadata = pd.read_csv(join(bd,"m1_patchseq_meta_data.csv"), delimiter="\t", index_col=1)

In [13]:
print(metadata)

                   Number             Slice        Date    Sample  \
Cell                                                                
20171204_sample_2       1  20171204_slice_2  2017-12-04  sample 2   
20171204_sample_4       2  20171204_slice_4  2017-12-04  sample 4   
20171204_sample_5       3  20171204_slice_5  2017-12-04  sample 5   
20171204_sample_6       4  20171204_slice_6  2017-12-04  sample 6   
20171207_sample_1       5  20171207_slice_1  2017-12-07  sample 1   
...                   ...               ...         ...       ...   
20200225_sample_2    1325  20200225_slice_2  2020-02-25  Sample 2   
20200225_sample_5    1326  20200225_slice_5  2020-02-25  Sample 5   
20200316_sample_1    1327  20200316_slice_1  2020-03-16  Sample 1   
20200316_sample_2    1328  20200316_slice_2  2020-03-16  Sample 2   
20200316_sample_3    1329  20200316_slice_3  2020-03-16  Sample 3   

                         Mouse Mouse date of birth  Mouse age Mouse gender  \
Cell                    

We standardize each electrophysiological feature and define the electrophysiological difference between cells as the Pearson’s correlation distance between their vectors of electrophysiological features.

In [16]:
import cajal.utilities

cells, gw_dist_dict = cajal.utilities.read_gw_dists(join(bd,'geodesic_100_gw.csv'),header=True)

common_cells = list(set(cells).intersection(exon_data.index))
metadata = metadata.loc[common_cells]
exon_data = exon_data.loc[common_cells]

In [8]:
import numpy as np

exon_data = np.log(5000 * exon_data + 1)
exon_dmat = 1-(exon_data.transpose().corr(method='pearson'))

In [80]:
import pandas as pd 

exon_data_df = pd.DataFrame(exon_data)

print(exon_data_df)
exon_data_df.shape

                   0610005C13Rik  0610006L08Rik  0610009B22Rik  0610009E02Rik  \
cell id                                                                         
20171204_sample_2       0.000000       0.000000       0.000000       0.000000   
20171204_sample_4       0.000000       0.000000      12.736704       0.000000   
20171204_sample_5       0.000000       0.000000      15.680366       0.000000   
20171204_sample_6       0.000000       0.000000       0.000000       0.000000   
20171207_sample_2       8.517393      11.082158       0.000000      12.899222   
...                          ...            ...            ...            ...   
20191114_sample_9       0.000000       0.000000       0.000000       0.000000   
20200106_sample_4       0.000000       0.000000       0.000000       0.000000   
20200225_sample_2       0.000000       0.000000       0.000000       0.000000   
20200225_sample_5       0.000000       0.000000      13.444448       8.517393   
20200316_sample_1       0.00

(645, 42466)

In [83]:
output_path = "/Users/apple/Desktop/KLin_Group/Project_2024/data/Was2CODE_analysis_turbo/exon_data_norm.csv"
exon_data_df.to_csv(output_path)

print(f"exon_data_norm saved to: {output_path}")

exon_data_norm saved to: /Users/apple/Desktop/KLin_Group/Project_2024/data/Was2CODE_analysis_turbo/exon_data_norm.csv


# Kevin's attempt

In [17]:
import os
from os.path import join
import numpy as np
import pandas as pd
from scipy.spatial.distance import squareform
import cajal.laplacian_score
import cajal.utilities

bd = "/Users/apple/Desktop/KLin_Group/Project_2024/data/Morpho_data/dataset/Scala" # Base directory

gw_dist = cajal.utilities.dist_mat_of_dict(gw_dist_dict, cells)
exon_data = pd.read_csv(join(bd,'exon_data.csv'),index_col='cell id')

# Align with the GW distance matrix by cells
exon_data = exon_data.loc[cells]

# Preprocess
exon_data = np.log(5000 * exon_data + 1)

# exon_data.shape = (N, G)
#  row: N cells
#  col: G genes



In [18]:
import copy

exon_data.shape

(645, 42466)

In [19]:
import numpy as np

# Compute column-wise variance
variances = exon_data.var()

# Define quantiles to compute (e.g., 0%, 25%, 50%, 75%, 100%)
quantiles = [0, 0.25, 0.5, 0.75, 1.0]

# Compute quantiles
quantile_values = np.quantile(variances, quantiles)

# Print results
for q, val in zip(quantiles, quantile_values):
    print(f"{int(q*100)}% quantile: {val}")


0% quantile: 0.0
25% quantile: 0.4478017940759585
50% quantile: 3.3048514713376393
75% quantile: 21.950672200076163
100% quantile: 54.744395907834196


In [24]:
import numpy as np

# Compute mean for each column
column_means = np.mean(exon_data_filtered, axis=0)

# Compute variance manually
column_variances = np.sum((exon_data_filtered - column_means) ** 2, axis=0) / (exon_data_filtered.shape[0] - 1)

# Print variances
print(column_variances)

0610012G03Rik    37.066848
1110004F10Rik    36.018177
1110032A03Rik    39.676500
1110038B12Rik    38.993792
1110038F14Rik    37.380421
                   ...    
Zup1             42.491318
Zw10             42.163913
Zyg11b           37.585229
Zyx              37.368209
Zzz3             40.784550
Length: 5041, dtype: float64


In [25]:
gw_dist

array([[  0.        ,  90.35402111,  57.43450838, ..., 111.30067322,
        158.42873872,  81.87195362],
       [ 90.35402111,   0.        ,  58.70649113, ...,  55.5314339 ,
         85.78594186, 125.12663229],
       [ 57.43450838,  58.70649113,   0.        , ...,  82.70654707,
        123.84900436,  97.48962517],
       ...,
       [111.30067322,  55.5314339 ,  82.70654707, ...,   0.        ,
         82.58141293, 159.82301257],
       [158.42873872,  85.78594186, 123.84900436, ...,  82.58141293,
          0.        , 194.51944195],
       [ 81.87195362, 125.12663229,  97.48962517, ..., 159.82301257,
        194.51944195,   0.        ]])

In [26]:
import pandas as pd 

gw_dist_df = pd.DataFrame(gw_dist)

print(gw_dist)
gw_dist_df.shape

[[  0.          90.35402111  57.43450838 ... 111.30067322 158.42873872
   81.87195362]
 [ 90.35402111   0.          58.70649113 ...  55.5314339   85.78594186
  125.12663229]
 [ 57.43450838  58.70649113   0.         ...  82.70654707 123.84900436
   97.48962517]
 ...
 [111.30067322  55.5314339   82.70654707 ...   0.          82.58141293
  159.82301257]
 [158.42873872  85.78594186 123.84900436 ...  82.58141293   0.
  194.51944195]
 [ 81.87195362 125.12663229  97.48962517 ... 159.82301257 194.51944195
    0.        ]]


(645, 645)

##### Turbo's generalized attempt （15000+ genes）

In [36]:
exon_data_filtered1 = exon_data.loc[:, exon_data.var() > 1]
print(exon_data_filtered1.shape)

(645, 27593)


In [37]:
import cajal.laplacian_score
import numpy as np
import pandas as pd
from scipy.spatial.distance import squareform

# NumPy array
if isinstance(gw_dist, pd.DataFrame):
    gw_dist_np1 = gw_dist.to_numpy()
else:
    gw_dist_np1 = gw_dist  

if isinstance(exon_data_filtered1, pd.DataFrame):
    exon_data_np1 = exon_data_filtered1.to_numpy()
else:
    exon_data_np1 = exon_data_filtered1


# laplacian_scores
np.random.seed(42)

laplacian1=cajal.laplacian_score.laplacian_scores(
    exon_data_np1,
    gw_dist_np1,
    np.median(squareform(gw_dist_np1)),
    permutations=5000,
    covariates=None,
    return_random_laplacians=False
)


In [38]:
import pandas as pd

laplacian_data1 = laplacian1[0]  

laplacian_df1 = pd.DataFrame(laplacian_data1)
laplacian_df1.index = exon_data_filtered1.columns

print(laplacian_df1.head())
laplacian_df1.shape

               feature_laplacians  laplacian_p_values  laplacian_q_values
0610005C13Rik            1.002784            0.720456            0.813651
0610009B22Rik            1.000980            0.240952            0.398095
0610009E02Rik            1.001666            0.397720            0.555323
0610009L18Rik            0.999815            0.096781            0.217474
0610010F05Rik            1.001851            0.450310            0.601089


(27593, 3)

In [39]:
output_path = "/Users/apple/Desktop/KLin_Group/Project_2024/data/Morpho_data/dataset/Scala/laplacian_scores_30000.csv"
laplacian_df1.to_csv(output_path)

print(f"Laplacian scores saved to: {output_path}")

Laplacian scores saved to: /Users/apple/Desktop/KLin_Group/Project_2024/data/Morpho_data/dataset/Scala/laplacian_scores_30000.csv


In [1]:
import pandas as pd
import numpy as np

# 1. Read data
print("Reading gene expression data...")
exon_data = pd.read_csv('/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/exon_data.csv', 
                        index_col=0)

print(f"Original data shape: {exon_data.shape}")
print(f"First few rows:\n{exon_data.iloc[:3, :3]}")

# 2. Calculate CPM (Counts Per Million) normalization
print("\nPerforming CPM normalization...")

# Calculate total counts per sample (column)
total_counts = exon_data.sum(axis=0)
print(f"Total counts per sample (first 5): {total_counts.head()}")

# CPM = (counts / total_counts) * 1,000,000
cpm_data = (exon_data / total_counts) * 1e6

print(f"CPM data range: {cpm_data.min().min()} to {cpm_data.max().max()}")

# 3. Apply log2 transformation: log2(CPM + 1)
print("\nApplying log2(CPM + 1) transformation...")
log2_cpm_data = np.log2(cpm_data + 1)

print(f"Log2(CPM+1) data shape: {log2_cpm_data.shape}")
print(f"Log2(CPM+1) data range: {log2_cpm_data.min().min():.3f} to {log2_cpm_data.max().max():.3f}")
print(f"Log2(CPM+1) data mean: {log2_cpm_data.mean().mean():.3f}")

# 4. Save normalized data
output_path = '/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/exon_norm_full.csv'
log2_cpm_data.to_csv(output_path)

print(f"\n✓ Normalized data saved to: {output_path}")
print(f"Final data dimensions: {log2_cpm_data.shape}")

Reading gene expression data...
Original data shape: (1329, 42466)
First few rows:
                   0610005C13Rik  0610006L08Rik  0610009B22Rik
cell id                                                       
20171204_sample_2              0              0              0
20171204_sample_4              0              0             68
20171204_sample_5              0              0           1291

Performing CPM normalization...
Total counts per sample (first 5): 0610005C13Rik     1201
0610006L08Rik       14
0610009B22Rik    55581
0610009E02Rik     2470
0610009L18Rik     6442
dtype: int64
CPM data range: 0.0 to 1000000.0

Applying log2(CPM + 1) transformation...
Log2(CPM+1) data shape: (1329, 42466)
Log2(CPM+1) data range: 0.000 to 19.932
Log2(CPM+1) data mean: 1.500

✓ Normalized data saved to: /home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/exon_norm_full.csv
Final data dimensions: (1329, 42466)


In [2]:
import pandas as pd
import numpy as np

# 检查gene expression数据
gex_df = pd.read_csv("/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/exon_norm_full.csv", 
                     header=0, index_col=0)
gex_data = gex_df.to_numpy()

print(f"Shape: {gex_data.shape}")
print(f"Contains NaN: {np.isnan(gex_data).any()}")
print(f"Contains Inf: {np.isinf(gex_data).any()}")
print(f"Min: {np.nanmin(gex_data)}")
print(f"Max: {np.nanmax(gex_data)}")

# 如果有NaN，找出有多少
if np.isnan(gex_data).any():
    nan_count = np.isnan(gex_data).sum()
    print(f"NaN count: {nan_count} / {gex_data.size} ({100*nan_count/gex_data.size:.2f}%)")

Shape: (1329, 42466)
Contains NaN: True
Contains Inf: False
Min: 0.0
Max: 19.931570012018494
NaN count: 357501 / 56437314 (0.63%)


## 2

In [2]:
"""
Minimal script to calculate 1-NN matching accuracy
"""
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Paths
gex_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/writeup22/cm_VAE_v6_2/outputs/attempt_1/images/coordinates/GEX_umap_coordinates_final.csv"
morpho_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/writeup22/cm_VAE_v6_2/outputs/attempt_1/images/coordinates/Morpho_umap_coordinates_final.csv"

# Load
gex = pd.read_csv(gex_path)
morpho = pd.read_csv(morpho_path)

# GEX -> Morpho
nn = NearestNeighbors(n_neighbors=1).fit(morpho[['Coord_1', 'Coord_2']])
idx = nn.kneighbors(gex[['Coord_1', 'Coord_2']], return_distance=False)
acc_gex = (gex['RNA_Family'].values == morpho['RNA_Family'].values[idx.flatten()]).mean()

# Morpho -> GEX
nn = NearestNeighbors(n_neighbors=1).fit(gex[['Coord_1', 'Coord_2']])
idx = nn.kneighbors(morpho[['Coord_1', 'Coord_2']], return_distance=False)
acc_morpho = (morpho['RNA_Family'].values == gex['RNA_Family'].values[idx.flatten()]).mean()

# Results
print(f"GEX -> Morpho accuracy: {acc_gex:.4f}")
print(f"Morpho -> GEX accuracy: {acc_morpho:.4f}")
print(f"Mean accuracy: {(acc_gex + acc_morpho)/2:.4f}")

GEX -> Morpho accuracy: 0.2453
Morpho -> GEX accuracy: 0.2093
Mean accuracy: 0.2273


## 3

In [3]:
"""
Minimal script to calculate 1-NN matching accuracy
"""
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Paths
gex_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/writeup22/cm_VAE_v6_3/outputs/attempt_1/images/coordinates/GEX_umap_coordinates_final.csv"
morpho_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/writeup22/cm_VAE_v6_3/outputs/attempt_1/images/coordinates/Morpho_umap_coordinates_final.csv"

# Load
gex = pd.read_csv(gex_path)
morpho = pd.read_csv(morpho_path)

# GEX -> Morpho
nn = NearestNeighbors(n_neighbors=1).fit(morpho[['Coord_1', 'Coord_2']])
idx = nn.kneighbors(gex[['Coord_1', 'Coord_2']], return_distance=False)
acc_gex = (gex['RNA_Family'].values == morpho['RNA_Family'].values[idx.flatten()]).mean()

# Morpho -> GEX
nn = NearestNeighbors(n_neighbors=1).fit(gex[['Coord_1', 'Coord_2']])
idx = nn.kneighbors(morpho[['Coord_1', 'Coord_2']], return_distance=False)
acc_morpho = (morpho['RNA_Family'].values == gex['RNA_Family'].values[idx.flatten()]).mean()

# Results
print(f"GEX -> Morpho accuracy: {acc_gex:.4f}")
print(f"Morpho -> GEX accuracy: {acc_morpho:.4f}")
print(f"Mean accuracy: {(acc_gex + acc_morpho)/2:.4f}")

GEX -> Morpho accuracy: 0.2295
Morpho -> GEX accuracy: 0.2155
Mean accuracy: 0.2225


## 6

In [1]:
"""
Minimal script to calculate 1-NN matching accuracy
"""
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Paths
gex_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/writeup22/cm_VAE_v6_6/outputs/attempt_1/images/coordinates/GEX_umap_coordinates_final.csv"
morpho_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/writeup22/cm_VAE_v6_6/outputs/attempt_1/images/coordinates/Morpho_umap_coordinates_final.csv"

# Load
gex = pd.read_csv(gex_path)
morpho = pd.read_csv(morpho_path)

# GEX -> Morpho
nn = NearestNeighbors(n_neighbors=1).fit(morpho[['Coord_1', 'Coord_2']])
idx = nn.kneighbors(gex[['Coord_1', 'Coord_2']], return_distance=False)
acc_gex = (gex['RNA_Family'].values == morpho['RNA_Family'].values[idx.flatten()]).mean()

# Morpho -> GEX
nn = NearestNeighbors(n_neighbors=1).fit(gex[['Coord_1', 'Coord_2']])
idx = nn.kneighbors(morpho[['Coord_1', 'Coord_2']], return_distance=False)
acc_morpho = (morpho['RNA_Family'].values == gex['RNA_Family'].values[idx.flatten()]).mean()

# Results
print(f"GEX -> Morpho accuracy: {acc_gex:.4f}")
print(f"Morpho -> GEX accuracy: {acc_morpho:.4f}")
print(f"Mean accuracy: {(acc_gex + acc_morpho)/2:.4f}")

GEX -> Morpho accuracy: 0.2385
Morpho -> GEX accuracy: 0.2620
Mean accuracy: 0.2503


## 12

In [4]:
"""
Minimal script to calculate 1-NN matching accuracy
"""
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Paths
gex_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/writeup22/cm_VAE_v6_12/outputs/attempt_1/images/coordinates/GEX_umap_coordinates_final.csv"
morpho_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/writeup22/cm_VAE_v6_12/outputs/attempt_1/images/coordinates/Morpho_umap_coordinates_final.csv"

# Load
gex = pd.read_csv(gex_path)
morpho = pd.read_csv(morpho_path)

# GEX -> Morpho
nn = NearestNeighbors(n_neighbors=1).fit(morpho[['Coord_1', 'Coord_2']])
idx = nn.kneighbors(gex[['Coord_1', 'Coord_2']], return_distance=False)
acc_gex = (gex['RNA_Family'].values == morpho['RNA_Family'].values[idx.flatten()]).mean()

# Morpho -> GEX
nn = NearestNeighbors(n_neighbors=1).fit(gex[['Coord_1', 'Coord_2']])
idx = nn.kneighbors(morpho[['Coord_1', 'Coord_2']], return_distance=False)
acc_morpho = (morpho['RNA_Family'].values == gex['RNA_Family'].values[idx.flatten()]).mean()

# Results
print(f"GEX -> Morpho accuracy: {acc_gex:.4f}")
print(f"Morpho -> GEX accuracy: {acc_morpho:.4f}")
print(f"Mean accuracy: {(acc_gex + acc_morpho)/2:.4f}")

GEX -> Morpho accuracy: 0.3115
Morpho -> GEX accuracy: 0.2465
Mean accuracy: 0.2790
